# Steam Game Recommendation System and Clustering

27,000 games on the Steam store, each with a description and the tags its players voted for. Given one game you like, which
others should you try? This notebook builds content-based recommenders from two different signals (what the developer **writes**
and what players **tag**), measures them against a popularity baseline without any user data, re-ranks them by quality, and groups
the whole catalogue into clusters you can name.

**Objectives**
1. What does the Steam catalogue look like, and how do we rank games by quality fairly?
2. How do we turn descriptions and tags into vectors (text normalization, Count vs TF-IDF)?
3. How do cosine similarity and the linear kernel find similar games, and when are they the same?
4. Without user histories, how can we tell whether recommendations are good?
5. Which signal recommends better: descriptions, tags, or both?
6. Can a quality score fix the recommender's taste for obscure games without drowning it in hits?
7. Which natural groups of games exist, and how many?

**Data** → [Kaggle dataset nikdavis/steam-store-games](https://www.kaggle.com/datasets/nikdavis/steam-store-games) · 27,075 games · store data up to May 2019

# [1/12] Setup

Set every knob once in the top cell → later cells only read them, and nothing is written unless `SAVE_FIGURES = True`

In [ ]:
"""Optional · Edit the settings, then run all cells. Explanations: README.md

[1/10] DATA_DIR: dataset folder (searched recursively for steam.csv if not found here)
    Default = '/kaggle/input/datasets/nikdavis/steam-store-games'
[2/10] WORK_DIR: folder for exported figures
    Default = '/kaggle/working'
[3/10] SEED: random seed
    Default = 42
[4/10] TOP_K: recommendations per game
    Default = 10
[5/10] MIN_RATINGS: a game needs at least this many ratings to be used as an evaluation query
    Default = 50
[6/10] N_QUERIES: games sampled as evaluation queries
    Default = 2000
[7/10] HYBRID_WEIGHT: weight of the tag signal in the hybrid (0 = descriptions only, 1 = tags only)
    Default = 0.5
[8/10] QUALITY_WEIGHTS: weights of the quality score tried when re-ranking
    Default = (0, 0.1, 0.2, 0.3, 0.5)
[9/10] K_RANGE: numbers of clusters tried
    Default = 2 ... 20
[10/10] SAVE_FIGURES: write every chart to WORK_DIR/assets
    Default = False
    For example:
        SAVE_FIGURES = True
"""
DATA_DIR        = '/kaggle/input/datasets/nikdavis/steam-store-games'
WORK_DIR        = '/kaggle/working'
SEED            = 42
TOP_K           = 10
MIN_RATINGS     = 50
N_QUERIES       = 2000
HYBRID_WEIGHT   = 0.5
QUALITY_WEIGHTS = (0, 0.1, 0.2, 0.3, 0.5)
K_RANGE         = range(2, 21)
SAVE_FIGURES    = False

EXAMPLE_GAMES    = ['Portal 2', 'Stardew Valley', 'Cities: Skylines', 'DARK SOULS™ III', 'Counter-Strike: Global Offensive']
WILSON_Z         = 1.96                  # 95 % confidence for the Wilson lower bound
TEXT_MIN_DF      = 5                     # a word must appear in at least 5 descriptions
TEXT_MAX_DF      = 0.5                   # ... and in at most half of them
TEXT_MAX_FEAT    = 50_000
SVD_COMPONENTS   = 100                   # tag vectors are compressed to 100 dimensions before clustering
SILHOUETTE_N     = 5000                  # games sampled to compute the silhouette score
TSNE_N           = 5000                  # games shown in the 2-D map
BATCH            = 500                   # queries scored per batch (keeps memory small)
PAD              = 32
PALETTE          = {'main': 'black', 'flag': 'red', 'soft': 'grey'}

In [ ]:
import re
import html
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from scipy import sparse

from nltk.stem import PorterStemmer
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, CountVectorizer, TfidfTransformer, TfidfVectorizer
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity, linear_kernel
from sklearn.preprocessing import normalize

warnings.filterwarnings('ignore')
sns.set_style('darkgrid')
sns.set_palette('gray')
rng = np.random.default_rng(SEED)
ASSETS = Path(WORK_DIR) / 'assets'
if SAVE_FIGURES:
    ASSETS.mkdir(parents=True, exist_ok=True)

In [ ]:
def show_block(title, items):
    print(f'{title}:')
    for key, value in items.items():
        print(f'    {key:<{PAD}}{value}')


def save(fig, name):
    if SAVE_FIGURES:
        fig.savefig(ASSETS / f'{name}.png', bbox_inches='tight', dpi=110)


def find_data_dir(default, root='/kaggle/input'):
    """Folder that holds steam.csv (with a steamspy_tags column). Tries `default`, then searches `root`."""
    import csv
    candidates = [Path(default)]
    if Path(root).exists():
        candidates += sorted(p.parent for p in Path(root).rglob('steam.csv'))
    for folder in candidates:
        if (folder / 'steam.csv').is_file():
            with open(folder / 'steam.csv', newline='', encoding='utf-8') as fh:
                if 'steamspy_tags' in next(csv.reader(fh)):
                    return folder
    mounted = sorted(str(p) for p in Path(root).glob('*/*'))[:20] if Path(root).exists() else []
    raise FileNotFoundError(f'(!) No steam.csv under {default} or {root}.\n'
                            f'    Add Input -> Datasets -> nikdavis/steam-store-games\n'
                            f'    Mounted now: {mounted or "nothing"}')

# [2/12] Data

Load the store table, the descriptions and the player tag votes, and align them on the game id → one row per game

In [ ]:
data_dir = find_data_dir(DATA_DIR)
store = pd.read_csv(data_dir / 'steam.csv')
descriptions = pd.read_csv(data_dir / 'steam_description_data.csv').rename(columns={'steam_appid': 'appid'})
tag_votes = pd.read_csv(data_dir / 'steamspy_tag_data.csv')

games = (store.merge(descriptions[['appid', 'short_description', 'about_the_game']], on='appid', how='inner')
              .merge(tag_votes[['appid']], on='appid', how='inner')
              .drop_duplicates('appid').reset_index(drop=True))
tag_matrix_df = tag_votes.set_index('appid').loc[games['appid']]
TAGS = tag_matrix_df.columns.tolist()
show_block('Inputs', {'data folder': data_dir, 'steam.csv': store.shape, 'descriptions': descriptions.shape,
                      'tag votes': tag_votes.shape})

In [ ]:
show_block('Outputs', {'games (all three sources)': len(games), 'tags': len(TAGS),
                       'games with no tag votes': int((tag_matrix_df.sum(axis=1) == 0).sum()),
                       'duplicate names (different ids)': int(games['name'].duplicated().sum())})
display(games[['appid', 'name', 'release_date', 'developer', 'genres', 'steamspy_tags', 'positive_ratings',
               'negative_ratings', 'owners', 'price']].head())

**Question:** What is in the data, and what has to be fixed before modelling?

**Answer:** Very little: all 27,075 store games have a description and player tag votes (371 tags), so nothing is lost when the
three files are joined. Only 1 game lacks a developer and 14 a publisher. Two things need converting:
- **owners** is a band ("20000-50000"), not a number → keep the lower bound as `owners_low`
- **ratings** come as positive and negative counts → total `ratings` and a quality score (section 4)

42 names appear twice under different ids (re-releases, regional editions); they stay, since they are different store pages.

In [ ]:
games['release_year'] = pd.to_datetime(games['release_date'], errors='coerce').dt.year
games['ratings'] = games['positive_ratings'] + games['negative_ratings']
games['owners_low'] = games['owners'].str.split('-').str[0].astype(int)
games['genre_set'] = games['genres'].fillna('').str.split(';').apply(lambda g: frozenset(x for x in g if x))
display(pd.DataFrame({'missing': games[['developer', 'publisher', 'genres', 'short_description', 'about_the_game']].isna().sum()}))
display(games['owners'].value_counts().rename('games').to_frame().head(13))

# [3/12] The catalogue

Count games per year, genre and owner band → the catalogue is young, indie-heavy and extremely unequal

In [ ]:
genre_counts = pd.Series([g for s in games['genre_set'] for g in s]).value_counts()
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
games['release_year'].value_counts().sort_index().loc[2000:2019].plot.bar(ax=axes[0], color=PALETTE['main'])
axes[0].set(title='Games released per year', xlabel='')
genre_counts.head(12).sort_values().plot.barh(ax=axes[1], color=PALETTE['main'])
axes[1].set(title='Most common genres')
order = games.groupby('owners')['owners_low'].first().sort_values().index
games['owners'].value_counts().reindex(order).plot.bar(ax=axes[2], color=PALETTE['main'])
axes[2].set(title='Games per owner band', xlabel='owners', yscale='log')
axes[2].tick_params(axis='x', rotation=60)
fig.tight_layout()
save(fig, '01_catalogue')
plt.show()

In [ ]:
show_block('Catalogue', {'released 2014 or later': f"{(games['release_year'] >= 2014).mean():.1%}",
                         'tagged Indie': f"{(genre_counts.get('Indie', 0) / len(games)):.1%}",
                         'below 20,000 owners': f"{(games['owners_low'] == 0).mean():.1%}",
                         'games with 0 ratings': int((games['ratings'] == 0).sum()),
                         'median ratings per game': int(games['ratings'].median())})

**Question:** What kind of catalogue are we recommending from?

**Answer:** a young, indie, long-tail catalogue:
- **Young** → 93.2 % of games were released in 2014 or later (after Steam Greenlight, 2012, opened the store to small developers)
- **Indie** → 71.7 % carry the Indie genre
- **Unequal** → 68.7 % have fewer than 20,000 owners, while 6 games have over 20 million; the median game has only 36 ratings

→ A recommender that follows popularity would show the same handful of hits to everyone; the interesting job is finding good games in the long tail.

# [4/12] A fair quality score

Rank games by the share of positive reviews, corrected for how many reviews they have → the Wilson lower bound

**Question:** Is a game with 5 positive reviews out of 5 better than one with 9,500 out of 10,000?

$$\text{Wilson}(p, n) = \frac{\hat p + \frac{z^2}{2n} - z\sqrt{\frac{\hat p(1-\hat p)}{n} + \frac{z^2}{4n^2}}}{1 + \frac{z^2}{n}}$$
- $\hat p$ → share of positive ratings, $n$ → number of ratings, $z = 1.96$ → 95 % confidence
- the result is the **lowest** positive share we can be 95 % sure of: few reviews → wide uncertainty → low score
- Analogy → a restaurant with one 5-star review vs one with 2,000 reviews averaging 4.7: the second is the safer bet

**Answer:** No. 5 positive out of 5 gives a Wilson score of only **0.566**, while 9,500 out of 10,000 gives **0.946**: the small
sample could easily be luck. Ranked by raw positive share, the top 5 are obscure games with about 100 reviews and none negative; ranked
by Wilson, the top includes *Portal 2* (138,220 positive, 98.7 %) and *Factorio* (47,918, 98.5 %).
→ The Wilson score is used as the quality signal for re-ranking.

In [ ]:
def wilson(pos, n, z=WILSON_Z):
    n = np.asarray(n, dtype=float)
    p = np.divide(pos, n, out=np.zeros_like(n), where=n > 0)
    centre = p + z ** 2 / (2 * n.clip(min=1))
    margin = z * np.sqrt(p * (1 - p) / n.clip(min=1) + z ** 2 / (4 * n.clip(min=1) ** 2))
    return np.where(n > 0, (centre - margin) / (1 + z ** 2 / n.clip(min=1)), 0.0)


games['positive_share'] = games['positive_ratings'] / games['ratings'].replace(0, np.nan)
games['wilson'] = wilson(games['positive_ratings'], games['ratings'])
examples = pd.DataFrame({'positive': [5, 9500, 90], 'ratings': [5, 10000, 100]})
examples['positive share'] = examples['positive'] / examples['ratings']
examples['wilson'] = wilson(examples['positive'], examples['ratings'])
display(examples.round(3))

In [ ]:
cols = ['name', 'positive_ratings', 'negative_ratings', 'positive_share', 'wilson']
display(pd.concat({'top by positive share': games.sort_values(['positive_share', 'ratings'], ascending=False)[cols].head(5).reset_index(drop=True),
                   'top by Wilson': games.sort_values('wilson', ascending=False)[cols].head(5).reset_index(drop=True)}).round(3))

# [5/12] Text normalization

Clean each description before counting words: strip HTML, lowercase, drop punctuation and stopwords, stem → fewer, more meaningful tokens

**Question:** Why normalize text, and what does each step do?

- **HTML / entities** → descriptions are store pages: `<br>`, `&quot;` and image tags are noise
- **Lowercase + punctuation** → "Shooter", "shooter," and "SHOOTER!" become one token
- **Stopwords** → "the", "and", "you" appear everywhere and say nothing about a game (scikit-learn's list of 318 English words)
- **Stemming (Porter)** → "build", "builds", "building" → `build`; crude but language-agnostic
- Analogy → sorting a library by topic: you ignore the font, the capital letters and the word "the" on every cover

**Answer:** Normalization halves the text and shrinks the vocabulary: the median description drops from 250 to 115 tokens and the
vocabulary from 89,918 to 61,950 distinct words. For *Portal 2*, 1,869 characters of store HTML become 140 tokens such as `test`,
`chamber`, `puzzl`, `co op`. 29 descriptions are empty after cleaning: they are written in Russian, Chinese, Japanese or Arabic, which this English tokenizer
drops entirely. Those games are still described by their tag vectors.

In [ ]:
STEMMER = PorterStemmer()
TOKEN = re.compile(r'[a-z][a-z0-9]+')
STOP = set(ENGLISH_STOP_WORDS)
TAG_RE = re.compile(r'<[^>]+>')


def normalize_text(text):
    text = html.unescape(TAG_RE.sub(' ', str(text))).lower()
    return ' '.join(STEMMER.stem(w) for w in TOKEN.findall(text) if w not in STOP)


raw_text = (games['short_description'].fillna('') + ' ' + games['about_the_game'].fillna(''))
games['clean_text'] = raw_text.map(normalize_text)
sample_i = games.index[games['name'] == 'Portal 2'][0]
show_block('Example: Portal 2', {'raw characters': len(raw_text[sample_i]),
                                 'clean tokens': len(games.loc[sample_i, 'clean_text'].split())})
print(raw_text[sample_i][:300], '...\n')
print(games.loc[sample_i, 'clean_text'][:300], '...')

In [ ]:
raw_tokens = raw_text.str.lower().str.findall(r'[a-z]+').map(len)
clean_tokens = games['clean_text'].str.split().map(len)
show_block('Effect of normalization', {'median tokens raw': int(raw_tokens.median()),
                                       'median tokens clean': int(clean_tokens.median()),
                                       'vocabulary raw': len(set(w for t in raw_text.str.lower().str.findall(r'[a-z]+') for w in t)),
                                       'vocabulary clean': len(set(w for t in games['clean_text'].str.split() for w in t)),
                                       'empty after cleaning': int((clean_tokens == 0).sum())})

# [6/12] Vectors: Count vs TF-IDF, text vs tags

Turn every game into a vector three ways → word counts, TF-IDF word weights, and TF-IDF weighted player tag votes

**Question:** How does TF-IDF decide which words matter?

$$\text{tfidf}(w, d) = \text{tf}(w, d) \cdot \log\frac{1 + N}{1 + \text{df}(w)} + 1 \qquad\qquad \vec v_d \leftarrow \frac{\vec v_d}{\lVert \vec v_d \rVert_2}$$
- $\text{tf}(w, d)$ → how often word $w$ appears in game $d$ (here $1 + \log \text{tf}$, so 50 mentions are not 50× one mention)
- $\text{df}(w)$ → in how many of the $N$ games it appears: a word in every description gets a weight near the minimum
- **Count** vectors keep raw counts → long descriptions and common words dominate
- **Tags** → the same idea applied to player votes: a game with 2,000 "Roguelike" votes is very roguelike, and a tag on every game ("Indie") counts less
- Analogy → a keyword that appears in only a few books is a better index entry than "chapter"

**Answer:** For *Portal 2* the heaviest TF-IDF words are `test chamber`, `portal`, `edit tool`, `apertur` (Aperture) — words almost
unique to this game — while common words like "game" or "play" get tiny weights. Its heaviest tags are science, online co-op, comedy and
first-person. Both vectors describe the game well but from different angles: the developer's words vs the players' labels.

In [ ]:
count_vec = CountVectorizer(min_df=TEXT_MIN_DF, max_df=TEXT_MAX_DF, max_features=TEXT_MAX_FEAT, ngram_range=(1, 2))
X_count = count_vec.fit_transform(games['clean_text']).astype(np.float32)
tfidf_vec = TfidfVectorizer(min_df=TEXT_MIN_DF, max_df=TEXT_MAX_DF, max_features=TEXT_MAX_FEAT, ngram_range=(1, 2),
                            sublinear_tf=True)
X_text = tfidf_vec.fit_transform(games['clean_text']).astype(np.float32)
X_tags = TfidfTransformer(sublinear_tf=True).fit_transform(sparse.csr_matrix(tag_matrix_df.to_numpy(dtype=np.float32)))
X_tags = normalize(X_tags).astype(np.float32)

vocab = np.array(tfidf_vec.get_feature_names_out())
row = X_text[sample_i].toarray().ravel()
top_words = pd.Series(row, index=vocab).sort_values(ascending=False).head(10)
tag_row = pd.Series(X_tags[sample_i].toarray().ravel(), index=TAGS).sort_values(ascending=False).head(10)
show_block('Shapes', {'Count (words)': X_count.shape, 'TF-IDF (words)': X_text.shape, 'TF-IDF (tags)': X_tags.shape})

In [ ]:
display(pd.DataFrame({'top TF-IDF words': top_words.index, 'weight': top_words.values.round(3),
                      'top tags': tag_row.index, 'tag weight': tag_row.values.round(3)}))

# [7/12] Similarity: cosine vs linear kernel

Measure how alike two games are with the angle between their vectors → for unit-length vectors the cosine and the dot product are the same number

**Question:** Cosine similarity or linear kernel — does it matter?

$$\cos(\vec a, \vec b) = \frac{\vec a \cdot \vec b}{\lVert \vec a\rVert\,\lVert \vec b\rVert} \qquad\qquad \text{linear}(\vec a, \vec b) = \vec a \cdot \vec b$$
- TF-IDF vectors are already scaled to length 1, so the denominator is 1 and **both give identical scores**; the linear kernel just skips the division
- **Count** vectors are not normalized: the dot product rewards long descriptions, the cosine does not
- Analogy → comparing two recipes by their ingredient proportions (cosine) instead of the total weight of the pot (dot product)

**Answer:** For TF-IDF vectors it does not matter: on 300 random games the largest difference between cosine and linear kernel is
7e-9 (rounding) and the top-10 lists are 100 % identical, so the faster linear kernel is used everywhere.
For **Count** vectors it matters a lot: only 13.7 % of the top-10 lists agree. The linear kernel recommends games with very long
descriptions (median 859 words, against 146 in the catalogue), while cosine stays near normal length (189).

In [ ]:
probe = rng.choice(len(games), 300, replace=False)
cos_tfidf = cosine_similarity(X_text[probe], X_text)
lin_tfidf = linear_kernel(X_text[probe], X_text)
cos_count = cosine_similarity(X_count[probe], X_count)
lin_count = linear_kernel(X_count[probe], X_count)


def top_k_overlap(a, b, k=TOP_K):
    ta, tb = np.argsort(-a, axis=1)[:, 1:k + 1], np.argsort(-b, axis=1)[:, 1:k + 1]
    return np.mean([len(set(x) & set(y)) / k for x, y in zip(ta, tb)])


text_len = np.asarray(X_count.sum(axis=1)).ravel()
lin_top = np.argsort(-lin_count, axis=1)[:, 1:TOP_K + 1]
cos_top = np.argsort(-cos_count, axis=1)[:, 1:TOP_K + 1]
show_block('300 random games', {
    'TF-IDF: max |cosine - linear|': f'{np.abs(cos_tfidf - lin_tfidf).max():.2e}',
    'TF-IDF: same top-10': f'{top_k_overlap(cos_tfidf, lin_tfidf):.1%}',
    'Count: same top-10': f'{top_k_overlap(cos_count, lin_count):.1%}',
    'median words, catalogue': int(np.median(text_len)),
    'median words, Count-linear top-10': int(np.median(text_len[lin_top])),
    'median words, Count-cosine top-10': int(np.median(text_len[cos_top])),
})

# [8/12] Offline evaluation without users

Score every recommender on 2,000 query games with signals it did not use → relevance, coverage, popularity bias and diversity

**Question:** How can recommendations be judged without knowing what users played?

For a query game $q$ and its top-$K$ list $R_q$:
$$\text{Genre precision@K} = \frac{1}{K}\sum_{r \in R_q} \frac{|G_q \cap G_r|}{|G_q \cup G_r|} \qquad \text{Coverage} = \frac{|\bigcup_q R_q|}{N} \qquad \text{ILD} = 1 - \overline{\cos}(r_i, r_j)$$
- **Genre precision** → average Jaccard overlap of the store genres $G$ (set by the developer, not used by the description model)
- **Same developer @K** → share of recommendations from the query's developer: a strong hint the games really are related
- **Coverage** → share of the catalogue that appears in at least one list: a recommender that only shows 200 games is useless for discovery
- **Popularity bias** → median owners (lower band) of the recommended games
- **ILD (intra-list diversity)** → how different the recommended games are from each other, measured on tags
- Analogy → judging a librarian without asking readers: do the suggested books share the shelf, the author, and do they cover more than the bestseller table?

**Answer:** content-based recommenders clearly beat both baselines, and **tags** beat descriptions:
- **Genre precision@10** → tags 0.645 and hybrid 0.645 vs descriptions 0.460 (TF-IDF) and 0.432 (Count); random lists score 0.252 and the
  popularity list only 0.215 (hits rarely share your game's genres)
- **Same developer@10** → TF-IDF descriptions 0.076, hybrid 0.071, tags 0.053: developers reuse wording and series names, so text finds
  sequels and siblings best
- **Coverage** → content models show 42-45 % of the catalogue; popularity shows 11 games (0.04 %)
- **Quality** → popularity lists are well reviewed (mean Wilson 0.818), content lists less so (0.556-0.604) → section 9

(!) Store genres and player tags overlap (tags often contain genre words), so the tag model's genre score is partly self-confirming,
and so is its low tag-based diversity (0.308). The description model is judged on fully independent signals. TF-IDF beats Count on
every relevance measure. The hybrid keeps the tag model's genre precision and most of the description model's developer hits,
so it is used from here on.

In [ ]:
eligible = np.flatnonzero(games['ratings'].to_numpy() >= MIN_RATINGS)
queries = rng.choice(eligible, size=min(N_QUERIES, len(eligible)), replace=False)
genre_sets = games['genre_set'].to_numpy()
developers = games['developer'].fillna('').to_numpy()
owners_low = games['owners_low'].to_numpy()
quality = games['wilson'].to_numpy()

w = HYBRID_WEIGHT
X_hybrid = sparse.hstack([np.sqrt(w) * X_tags, np.sqrt(1 - w) * X_text]).tocsr()   # cos = w*cos_tags + (1-w)*cos_text
REPRESENTATIONS = {'Descriptions (Count)': normalize(X_count), 'Descriptions (TF-IDF)': X_text,
                   'Tags (TF-IDF)': X_tags, f'Hybrid (tags {w:.0%} + text)': X_hybrid}


def recommend_scores(X, rows):
    return np.asarray(linear_kernel(X[rows], X), dtype=np.float32)


def top_k(scores, rows, k=TOP_K):
    scores = scores.copy()
    scores[np.arange(len(rows)), rows] = -np.inf               # never recommend the query itself
    part = np.argpartition(-scores, k, axis=1)[:, :k]
    order = np.argsort(-np.take_along_axis(scores, part, axis=1), axis=1)
    return np.take_along_axis(part, order, axis=1)


def jaccard(a, b):
    return len(a & b) / len(a | b) if (a or b) else 0.0


def evaluate(name, lists):
    gp = np.mean([np.mean([jaccard(genre_sets[q], genre_sets[r]) for r in rec]) for q, rec in zip(queries, lists)])
    dev = np.mean([np.mean([developers[r] == developers[q] and developers[q] != '' for r in rec]) for q, rec in zip(queries, lists)])
    ild = np.mean([1 - (lambda s: (s.sum() - len(rec)) / (len(rec) * (len(rec) - 1)))(
        (X_tags[rec] @ X_tags[rec].T).toarray()) for rec in lists])
    return {'recommender': name, 'genre precision@10': gp, 'same developer@10': dev,
            'coverage': len(np.unique(lists)) / len(games), 'median owners (low band)': int(np.median(owners_low[lists])),
            'mean Wilson': float(np.mean(quality[lists])), 'diversity (ILD)': ild}


lists_by_model, rows_out = {}, []
pop_order = np.argsort(-games['ratings'].to_numpy())
lists_by_model['Popularity (most rated)'] = np.array([[g for g in pop_order[:TOP_K + 1] if g != q][:TOP_K] for q in queries])
lists_by_model['Random'] = np.array([rng.choice(np.delete(np.arange(len(games)), q), TOP_K, replace=False) for q in queries])
for i, (name, X) in enumerate(REPRESENTATIONS.items(), 1):
    print(f'[{i}/{len(REPRESENTATIONS)}] {name}')
    lists_by_model[name] = np.vstack([top_k(recommend_scores(X, queries[s:s + BATCH]), queries[s:s + BATCH])
                                      for s in range(0, len(queries), BATCH)])
for name, lists in lists_by_model.items():
    rows_out.append(evaluate(name, lists))
leaderboard = pd.DataFrame(rows_out).set_index('recommender')

In [ ]:
display(leaderboard.round(3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 4.8))
for ax, col in zip(axes, ['genre precision@10', 'same developer@10', 'coverage']):
    s = leaderboard[col].sort_values()
    ax.barh(s.index, s.values, color=[PALETTE['flag'] if 'Hybrid' in i else PALETTE['main'] for i in s.index])
    ax.set(title=col)
fig.tight_layout()
save(fig, '02_leaderboard')
plt.show()

# [9/12] Re-ranking by quality

Blend similarity with the Wilson score → recommendations move towards well-reviewed games, and we measure what it costs

**Question:** Can we prefer good games without turning the recommender into a bestseller list?

$$\text{score}(q, r) = (1 - \beta)\,\cos(q, r) + \beta\,\text{Wilson}(r)$$
- $\beta = 0$ → pure similarity; $\beta = 1$ → pure quality ranking (the same list for everyone)
- re-ranking happens inside the 100 most similar games, so a recommendation is always somewhat relevant

**Answer:** yes, with a small price. At $\beta = 0.2$ the mean Wilson of the recommendations rises from 0.604 to **0.799** (close to the
popularity list's 0.818) while genre precision only drops from 0.645 to 0.624. The cost is coverage: 42.2 % → 30.8 % of the catalogue,
because weak games stop being shown. At $\beta = 0.5$ quality barely improves further (0.871) while coverage falls to 18.9 %.
→ $\beta = 0.2$ is used for the final recommender.

In [ ]:
HYBRID = f'Hybrid (tags {w:.0%} + text)'
X_best = REPRESENTATIONS[HYBRID]
POOL = 100
rerank_rows = []
for beta in QUALITY_WEIGHTS:
    lists = []
    for s in range(0, len(queries), BATCH):
        rows = queries[s:s + BATCH]
        sim = recommend_scores(X_best, rows)
        pool = top_k(sim, rows, POOL)
        blended = (1 - beta) * np.take_along_axis(sim, pool, axis=1) + beta * quality[pool]
        lists.append(np.take_along_axis(pool, np.argsort(-blended, axis=1)[:, :TOP_K], axis=1))
    rerank_rows.append({'beta': beta, **evaluate(f'beta = {beta}', np.vstack(lists))})
rerank = pd.DataFrame(rerank_rows).drop(columns='recommender').set_index('beta')

In [ ]:
display(rerank.round(3))

# [10/12] Clustering the catalogue

Compress the tag vectors with SVD, run k-means for k = 2 … 20, choose k with the elbow and the silhouette → name each cluster by its top tags

**Question:** How many natural groups of games are there?

$$\text{inertia} = \sum_{i} \lVert x_i - \mu_{c(i)} \rVert^2 \qquad\qquad s(i) = \frac{b(i) - a(i)}{\max\{a(i), b(i)\}}$$
- **Elbow** → inertia always falls as $k$ grows; look for the $k$ after which it falls much more slowly
- **Silhouette** → $a(i)$ = mean distance to its own cluster, $b(i)$ = to the nearest other cluster; +1 = clearly placed, 0 = on a border
- **SVD first** → 371 sparse tag weights become 100 dense dimensions (latent semantic analysis): k-means distances behave better
- **Why not the old features?** → my first version clustered on scaled numbers plus label-encoded developer and publisher ids:
  "developer 4,211 is closer to developer 4,212 than to developer 12" is meaningless, so the clusters were too
- Analogy → sorting a record collection: by genre labels it forms piles; by catalogue number it forms nonsense

**Answer:** there is **no strong natural number** of groups. Inertia falls smoothly with no clear elbow, and the silhouette stays
low everywhere (0.04 at k = 2 rising to 0.10 at k = 18-20): games blend into each other (an indie action-adventure with RPG elements
belongs to several groups). 100 SVD dimensions keep 80.9 % of the tag variance. The best silhouette with at least 4 clusters is
k = 18, which gives clusters small enough to name.

In [ ]:
svd = TruncatedSVD(n_components=SVD_COMPONENTS, random_state=SEED)
Z = normalize(svd.fit_transform(X_tags))
sil_idx = rng.choice(len(Z), SILHOUETTE_N, replace=False)
k_rows = []
for k in K_RANGE:
    km = KMeans(n_clusters=k, n_init=4, random_state=SEED).fit(Z)
    k_rows.append({'k': k, 'inertia': km.inertia_, 'silhouette': silhouette_score(Z[sil_idx], km.labels_[sil_idx])})
k_table = pd.DataFrame(k_rows).set_index('k')

fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
axes[0].plot(k_table.index, k_table['inertia'], marker='o', color=PALETTE['main'])
axes[0].set(title='Elbow: inertia vs k', xlabel='k')
axes[1].plot(k_table.index, k_table['silhouette'], marker='o', color=PALETTE['flag'])
axes[1].set(title='Silhouette vs k', xlabel='k')
fig.tight_layout()
save(fig, '03_choose_k')
plt.show()

In [ ]:
show_block('SVD', {'variance kept by 100 components': f'{svd.explained_variance_ratio_.sum():.1%}'})
display(k_table.round(3).T)

**Question:** What are the clusters?

**Answer:** the 18 clusters read like Steam store shelves: strategy (turn-based, RTS), FPS / multiplayer shooters, shoot-'em-up /
bullet hell, RPGs (including RPG Maker games), platformers, puzzle, horror, point-and-click adventures, visual novels, VR, racing,
sports, tower defense, free-to-play MMOs, violent / gore action, simulation and a large casual / hidden-object group.
Quality differs by shelf: the casual / hidden-object cluster has a median Wilson of 0.371 and the Early Access action cluster 0.376,
while visual novels (0.714), FPS (0.690) and strategy (0.685) are the best reviewed. The t-SNE map shows separate islands for niches —
visual novels (17), racing (15), sports (6), point-and-click (4), tower defense (16) — and a large overlapping centre where FPS (10),
shoot-'em-ups (11), platformers (2), puzzle (14) and Early Access action (7) blend, matching the low silhouette.

In [ ]:
N_CLUSTERS = int(k_table['silhouette'].loc[4:].idxmax())          # best silhouette with at least 4 clusters
km = KMeans(n_clusters=N_CLUSTERS, n_init=10, random_state=SEED).fit(Z)
games['cluster'] = km.labels_
tag_dense = X_tags.toarray()
profiles = []
for c in range(N_CLUSTERS):
    members = games['cluster'] == c
    mean_tags = pd.Series(tag_dense[members.to_numpy()].mean(axis=0), index=TAGS).sort_values(ascending=False)
    top_games = games[members].sort_values('ratings', ascending=False)['name'].head(3).tolist()
    profiles.append({'cluster': c, 'games': int(members.sum()), 'top tags': ', '.join(mean_tags.index[:5]),
                     'median Wilson': round(float(games.loc[members, 'wilson'].median()), 3),
                     'best-known games': ' | '.join(top_games)})
profiles = pd.DataFrame(profiles).sort_values('games', ascending=False).set_index('cluster')

map_idx = rng.choice(len(Z), TSNE_N, replace=False)
xy = TSNE(n_components=2, perplexity=40, init='pca', random_state=SEED).fit_transform(Z[map_idx])
fig, ax = plt.subplots(figsize=(11, 8))
cmap = plt.get_cmap('tab20', N_CLUSTERS)
ax.scatter(xy[:, 0], xy[:, 1], c=games['cluster'].to_numpy()[map_idx], cmap=cmap, s=6, alpha=0.7)
for c in range(N_CLUSTERS):
    m = games['cluster'].to_numpy()[map_idx] == c
    if m.sum():
        ax.text(*np.median(xy[m], axis=0), str(c), fontsize=13, weight='bold', ha='center')
ax.set(title=f't-SNE map of {TSNE_N:,} games coloured by cluster (k = {N_CLUSTERS})', xticks=[], yticks=[])
save(fig, '04_cluster_map')
plt.show()

In [ ]:
show_block('Clusters', {'k chosen': N_CLUSTERS, 'silhouette (all-k best)': f"k = {int(k_table['silhouette'].idxmax())}"})
pd.set_option('display.max_colwidth', 90)
display(profiles)

# [11/12] Recommendations in action

Ask the final recommender (hybrid, quality-aware) for five well-known games → read the lists like a player would

In [ ]:
BEST_BETA = 0.2


def find_game(title):
    exact = games.index[games['name'].str.lower() == title.lower()]
    if len(exact):
        return games.loc[exact, 'ratings'].idxmax()
    hits = games[games['name'].str.lower().str.contains(re.escape(title.lower()))]
    return hits['ratings'].idxmax() if len(hits) else None


def recommend(title, k=TOP_K, beta=BEST_BETA):
    q = find_game(title)
    sim = recommend_scores(X_best, np.array([q]))
    pool = top_k(sim, np.array([q]), POOL)[0]
    blended = (1 - beta) * sim[0, pool] + beta * quality[pool]
    picks = pool[np.argsort(-blended)[:k]]
    out = games.loc[picks, ['name', 'developer', 'genres', 'wilson', 'cluster']].copy()
    out.insert(1, 'similarity', sim[0, picks].round(3))
    return games.loc[q, 'name'], out.reset_index(drop=True)


for title in EXAMPLE_GAMES:
    name, recs = recommend(title)
    print(f'\nBecause you liked {name}:')
    display(recs.head(TOP_K).round(3))

**Question:** Do the lists make sense to a player?

**Answer:** mostly yes, and where it fails the reason is visible:
- **Dark Souls III** → the two other Dark Souls games first, then Dragon's Dogma, Oblivion, Skyrim — action RPGs
- **Counter-Strike: Global Offensive** → Counter-Strike: Source and Condition Zero, Insurgency, Arma, Call of Duty — tactical shooters
- **Cities: Skylines** → Parkitect, Prison Architect, Transport Fever, SimCity 4, Planet Coaster — builders and management games
- **Stardew Valley** → My Time at Portia, Forager, Garden Paws, Starbound — farming and life sims
- **Portal 2** → *Portal* first, but then Serious Sam, Bulletstorm and Borderlands: the shared "first-person", "co-op" and "sci-fi" tags
  outweigh "puzzle". Puzzle games like *The Talos Principle* are missed — a limit of content similarity, which knows what a game is
  tagged with but not why people enjoy it.

# [12/12] Summary

**Answer:**
- **Data** → 27,075 games, a young, indie long tail: the median game has 36 ratings
- **Quality** → the Wilson lower bound ranks *Portal 2* and *Factorio* above games with 100 perfect reviews
- **Text** → normalization halves the tokens; TF-IDF beats raw counts, and for TF-IDF the cosine and linear kernel are identical
- **Best signal** → player tags (genre precision 0.645 vs 0.460 for descriptions); descriptions find more same-developer games
- **Baselines** → popularity reaches only 11 games and matches genres worse than random lists
- **Re-ranking** → $\beta = 0.2$ lifts mean quality from 0.604 to 0.799 for a 2-point drop in genre precision
- **Clusters** → 18 shelf-like groups, but a low silhouette (0.10): game genres overlap rather than forming clean islands
- **Next steps** → user data (play time, reviews) for collaborative filtering; a learned weighting of tags vs text; evaluation against
  real co-play or wishlist data instead of genre overlap